# Comparison with pyvinecopulib

Compare MLE fits of an unrotated Gumbel copula, then fit the same truncated
D-vine in both libraries on S&P 500 pseudo-observations. The final table
retains dimensions 10, 50, and 100, with fit/sampling times and likelihoods.
Fixing both structure and families makes differences easier to interpret.

Install the notebook and reference-library extras with
`pip install -e ".[examples,external]"` from the repository root.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import pyvinecopulib as pv

from pyscarcopula import GumbelCopula, VineCopula
from pyscarcopula._utils import pobs

## Data and controls

Both libraries receive the same ranked returns and asset order. Keep assets
with at least 95% price coverage, forward-fill remaining gaps, and remove
incomplete rows. Forward-filled prices can produce tied returns.

In [2]:
DIMS = [10, 50, 100]
TRUNCATION_LEVEL = 2
N_SAMPLE = 5_000
SEED = 2026

DATA_DIR = Path("data") if Path("data").exists() else Path("..") / "data"
prices = pd.read_csv(
    DATA_DIR / "sp500_yfinance_2020_prices.csv", parse_dates=["Date"]
).set_index("Date").apply(pd.to_numeric, errors="coerce")
prices = prices.dropna(axis=1, thresh=int(0.95 * len(prices))).ffill().dropna()
returns = np.log(prices / prices.shift(1)).dropna()
u_all = np.asfortranarray(pobs(returns.to_numpy()), dtype=np.float64)
if not DIMS or any(d < 2 or d > u_all.shape[1] for d in DIMS):
    raise ValueError(f"DIMS must contain dimensions from 2 to {u_all.shape[1]}")
if TRUNCATION_LEVEL < 1:
    raise ValueError("TRUNCATION_LEVEL must be at least 1")

print(f"Returns: {len(u_all)} rows, {u_all.shape[1]} assets")
print(f"Dimensions: {DIMS}; simulated observations per model: {N_SAMPLE}")

Returns: 1581 rows, 105 assets
Dimensions: [10, 50, 100]; simulated observations per model: 5000


## Bivariate fit and density

First estimate the parameter independently in each library. Then evaluate
both densities at one shared parameter to isolate numerical differences from
optimizer differences. The density check uses a log-density tolerance of
`1e-8 + 1e-8 * abs(reference)` on these observations; it is a numerical
sanity check, not a universal accuracy guarantee.

In [3]:
u2 = np.asfortranarray(u_all[:, :2])
ps_bi = GumbelCopula(rotate=0)
ps_result = ps_bi.fit(u2, method="mle")
pv_bi = pv.Bicop(family=pv.BicopFamily.gumbel, rotation=0)
pv_controls = pv.FitControlsBicop(parametric_method="mle")
pv_bi.fit(u2, controls=pv_controls)

print("Assets:", ", ".join(returns.columns[:2]))
pd.DataFrame({
    "parameter": [ps_result.copula_param, pv_bi.parameters[0, 0]],
    "loglik_per_obs": [ps_result.log_likelihood / len(u2), pv_bi.loglik(u2) / len(u2)],
}, index=["pyscarcopula", "pyvinecopulib"])

Assets: AAPL, ABBV


,parameter,loglik_per_obs
pyscarcopula,1.152247,0.028454
pyvinecopulib,1.152247,0.028454


In [4]:
theta = float(ps_result.copula_param)
pv_reference = pv.Bicop(
    family=pv.BicopFamily.gumbel, rotation=0, parameters=np.array([[theta]])
)
ps_logpdf = ps_bi.log_pdf(u2[:, 0], u2[:, 1], np.full(len(u2), theta))
pv_logpdf = np.log(pv_reference.pdf(u2))
pd.Series({
    "parameter_abs_diff": abs(theta - pv_bi.parameters[0, 0]),
    "shared_parameter_logpdf_max_abs_diff": np.max(np.abs(ps_logpdf - pv_logpdf)),
    "shared_parameter_density_matches": np.allclose(
        ps_logpdf, pv_logpdf, rtol=1e-8, atol=1e-8
    ),
})

parameter_abs_diff                       0.0
shared_parameter_logpdf_max_abs_diff     0.0
shared_parameter_density_matches        True
dtype: object

## Same D-vine, families, and truncation

The first tree is the path through the first `d` assets in column order.
Both libraries fit unrotated Gumbel edges in the active trees and independence
above the truncation level. Each model is fitted once, without structure or
family selection. The small decoder below checks semantic edges because raw
R-vine matrix conventions and index bases differ between libraries.

In [5]:
def timed(function):
    start = perf_counter()
    result = function()
    return result, perf_counter() - start


def pv_tree_edges(model, tree):
    structure = model.structure
    order = [int(v) - 1 for v in structure.order]
    return {
        (
            frozenset((order[edge], int(structure.struct_array(tree, edge, False)) - 1)),
            frozenset(int(structure.struct_array(k, edge, False)) - 1 for k in range(tree)),
        )
        for edge in range(model.dim - tree - 1)
    }

In [6]:
rows = []
for d in DIMS:
    u_d = np.asfortranarray(u_all[:, :d])
    levels = min(TRUNCATION_LEVEL, d - 1)
    ps_vine = VineCopula.dvine(d=d, order=range(d))
    fixed_specs = [[(GumbelCopula, 0) for _ in tree] for tree in ps_vine.structure.to_trees()]
    pv_vine = pv.Vinecop.from_structure(
        structure=pv.DVineStructure(list(range(1, d + 1))),
        pair_copulas=[
            [pv.Bicop(family=pv.BicopFamily.gumbel, rotation=0) for _ in range(d - tree - 1)]
            for tree in range(levels)
        ],
    )
    _, ps_fit_s = timed(lambda: ps_vine.fit(
        u_d, method="mle", copulas=fixed_specs, truncation_level=levels
    ))
    _, pv_fit_s = timed(lambda: pv_vine.fit(u_d, controls=pv_controls, num_threads=1))
    edges_match = all(
        set(ps_vine.structure.to_trees()[tree]) == pv_tree_edges(pv_vine, tree)
        for tree in range(levels)
    )
    assert edges_match, "The fitted models must have identical active tree edges"

    ps_ll = ps_vine.log_likelihood(u_d) / len(u_d)
    pv_ll = pv_vine.loglik(u_d) / len(u_d)
    ps_sample, ps_sample_s = timed(lambda: ps_vine.sample(
        N_SAMPLE, rng=np.random.default_rng(SEED + d)
    ))
    pv_sample, pv_sample_s = timed(lambda: pv_vine.simulate(N_SAMPLE, seeds=[SEED + d]))
    corr_diff = np.corrcoef(ps_sample, rowvar=False) - np.corrcoef(pv_sample, rowvar=False)
    rows.append({
        "d": d,
        "edges_match": edges_match,
        "ps_loglik_per_obs": ps_ll,
        "pv_loglik_per_obs": pv_ll,
        "loglik_diff_per_obs": ps_ll - pv_ll,
        "ps_fit_s": ps_fit_s,
        "pv_fit_s": pv_fit_s,
        "ps_sample_s": ps_sample_s,
        "pv_sample_s": pv_sample_s,
        "sample_corr_rmse": np.sqrt(np.mean(corr_diff[np.triu_indices(d, k=1)] ** 2)),
    })

comparison = pd.DataFrame(rows).set_index("d")
comparison

,edges_match,ps_loglik_per_obs,pv_loglik_per_obs,loglik_diff_per_obs,ps_fit_s,pv_fit_s,ps_sample_s,pv_sample_s,sample_corr_rmse
d,,,,,,,,,
10,True,1.620587,1.620587,1.220896e-08,0.053761,0.028649,0.025039,0.020542,0.016313
50,True,9.649014,9.649014,-7.597269e-08,0.487016,0.167834,0.136213,0.127316,0.019361
100,True,15.282571,15.282571,7.237517e-08,1.682993,0.338917,0.275224,0.250213,0.019683


## Reading the comparison

- `ps` and `pv` denote `pyscarcopula` and `pyvinecopulib`. `edges_match` must
  be true before interpreting their likelihood difference. A difference near
  zero indicates agreement of the fitted likelihoods for this dataset.
- `sample_corr_rmse` compares off-diagonal correlation coefficients of the
  two simulated uniform samples. It checks pairwise dependence, not full
  distributional equality. Monte Carlo variation remains even with the same
  seed because the libraries use different random-number implementations.
- All dimensions use the same sample size. Times are single-run wall-clock
  observations, excluding model construction and input conversion. They can
  include first-call overhead and should not be read as stable speed rankings.
- This is a fixed-structure Gumbel comparison. It does not establish agreement
  of automatic structure/family selection or dynamic GAS/SCAR estimators.